In [2]:

import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
nltk.download('vader_lexicon')


data = {
    'review': [
        'This product is amazing I love it', 'Worst purchase ever waste of money',
        'Average product nothing special', 'Excellent quality highly recommend',
        'Terrible experience will not buy again', 'It is okay does the job',
        'Best product in market fantastic', 'Poor quality broke in 2 days',
        'Good value for money satisfied', 'Not bad not good just average'
    ] * 50,
    'overall': [5,1,3,5,1,3,5,1,4,3] * 50
}
df = pd.DataFrame(data)

# Pre-processing + Sentiment Categories
def get_sentiment(rating):
    if rating <= 2: return 'Negative'
    elif rating == 3: return 'Neutral'
    else: return 'Positive'
df['sentiment'] = df['overall'].apply(get_sentiment)

# VADER Polarity
sia = SentimentIntensityAnalyzer()
df['vader_compound'] = df['review'].apply(lambda x: sia.polarity_scores(x)['compound'])
print("VADER Scores Added")
print(df[['review','vader_compound','sentiment']].head())

# Decision Tree Model
X = df['review']
y = df['sentiment']
vectorizer = CountVectorizer()
X_vec = vectorizer.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
print("\nDecision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print(confusion_matrix(y_test, dt_pred))

# Naive Bayes Model
nb = MultinomialNB()
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)
print("\nNaive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print(confusion_matrix(y_test, nb_pred))

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


VADER Scores Added
                                   review  vader_compound sentiment
0       This product is amazing I love it          0.8402  Positive
1      Worst purchase ever waste of money         -0.7845  Negative
2         Average product nothing special         -0.3089   Neutral
3      Excellent quality highly recommend          0.7574  Positive
4  Terrible experience will not buy again         -0.4767  Negative

Decision Tree Accuracy: 1.0
[[35  0  0]
 [ 0 28  0]
 [ 0  0 37]]

Naive Bayes Accuracy: 1.0
[[35  0  0]
 [ 0 28  0]
 [ 0  0 37]]


In [3]:

import zipfile, os
os.makedirs('company_data', exist_ok=True)
text1 = "Infosys Limited is a public company. Headquarters: Bangalore. Domain: IT Services. Incorporated: 1981."
text2 = "TCS is a private company. Headquarters: Mumbai. Domain: Consulting. Incorporated: 1968."
text3 = "Wipro Limited is public. Headquarters: Bangalore. Domain: Software. Incorporated: 1945."

with open('company_data/file1.txt','w') as f: f.write(text1)
with open('company_data/file2.txt','w') as f: f.write(text2)
with open('company_data/file3.txt','w') as f: f.write(text3)

# Read + Preprocess
import glob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import spacy
nlp = spacy.load('en_core_web_sm')

all_text = []
for file in glob.glob('company_data/*.txt'):
    with open(file,'r') as f: all_text.append(f.read())

# NER - Company, Headquarters Extract
for text in all_text:
    doc = nlp(text)
    print("\nEntities:", [(ent.text, ent.label_) for ent in doc.ents if ent.label_ in ['ORG','GPE']])

# LDA Topic Modelling
tfidf = TfidfVectorizer(stop_words='english')
dtm = tfidf.fit_transform(all_text)
lda = LatentDirichletAllocation(n_components=2, random_state=42)
lda.fit(dtm)

for i, topic in enumerate(lda.components_):
    print(f"\nTopic {i}:")
    print([tfidf.get_feature_names_out()[index] for index in topic.argsort()[-5:]])


Entities: [('Infosys Limited', 'ORG')]

Entities: [('Wipro Limited', 'ORG')]

Entities: [('TCS', 'ORG'), ('Mumbai', 'GPE')]

Topic 0:
['tcs', 'mumbai', 'consulting', '1968', 'private']

Topic 1:
['incorporated', 'domain', 'bangalore', 'public', 'limited']
